<a href="https://colab.research.google.com/github/linhkid/pallas-forge/blob/main/notebooks/04_auto_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Auto-Tuning Pallas Kernels

This notebook demonstrates the `pallas-forge` auto-tuning framework:

1. Define a configuration search space
2. Run systematic benchmarks across all configs
3. Generate performance heatmaps
4. Analyze results to understand *why* some configs are faster

> **Note**: On CPU (interpret mode), timing differences are minimal because there's no real
> MXU/VMEM hardware. On TPU, you'd see 3-5x swings between configs. The framework and
> visualization pipeline work identically regardless.

In [1]:
# ============================================================
# Colab / TPU Setup — run this cell first!
# ============================================================
# On Colab: Runtime > Change runtime type > TPU
# Skip this cell if running locally with pallas-forge already installed.
# ============================================================

import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "jax[tpu]", "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ]
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pallas-forge[viz] @ git+https://github.com/linhkid/pallas-forge.git@v0.1.2",
        ]
    )
    print("Colab setup complete!")
else:
    print("Running locally — skipping Colab install.")

Colab setup complete!


In [2]:
import jax
import jax.numpy as jnp
import matplotlib

matplotlib.use("Agg")  # Use non-interactive backend
import matplotlib.pyplot as plt

from pallas_forge import fused_swiglu, tiled_matmul
from pallas_forge.tune import TuneConfig, tune

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


## 1. Define the search space

`TuneConfig` defines which parameter combinations to explore.

In [3]:
# From a Python dict.
# TPU constraint: block_k and block_n must be multiples of 128. block_m can be
# any multiple of 8, but in practice >= 128 gives better MXU utilization.
config = TuneConfig.from_dict(
    {
        "block_m": [128, 256, 512],
        "block_k": [128, 256],
        "block_n": [128, 256, 512],
    }
)

print(f"Parameters: {config.param_names}")
print(f"Total combinations: {config.total_combinations}")

# Add constraints to prune combinations that would spill VMEM on v5e.
# Rule of thumb: block_m * block_n * 4 bytes (fp32 accumulator) <= ~16 MiB.
config.add_constraint(lambda p: p["block_m"] * p["block_n"] <= 512 * 512)

valid_configs = config.grid()
print(f"Valid combinations (after constraints): {len(valid_configs)}")
print("\nFirst 5 configs:")
for c in valid_configs[:5]:
    print(f"  {c}")

Parameters: ['block_m', 'block_k', 'block_n']
Total combinations: 18
Valid combinations (after constraints): 18

First 5 configs:
  {'block_m': 128, 'block_k': 128, 'block_n': 128}
  {'block_m': 128, 'block_k': 128, 'block_n': 256}
  {'block_m': 128, 'block_k': 128, 'block_n': 512}
  {'block_m': 128, 'block_k': 256, 'block_n': 128}
  {'block_m': 128, 'block_k': 256, 'block_n': 256}


## 2. Run auto-tuning with `tune()`

The `tune()` function is the single entry point. It:
1. Generates configs from the search space
2. Benchmarks each one (with warmup + statistical timing)
3. Returns a `TuneReport` for analysis

In [4]:
# Problem setup — large enough that tile choices actually vary.
M, K, N = 1024, 1024, 1024


def input_fn(cfg):
    """Create inputs for each config."""
    key = jax.random.PRNGKey(0)
    k1, k2 = jax.random.split(key)
    x = jax.random.normal(k1, (M, K), dtype=jnp.float32)
    w = jax.random.normal(k2, (K, N), dtype=jnp.float32)
    return (x, w)


def kernel_fn(x, w, *, block_m, block_k, block_n):
    """Kernel wrapper matching tune() calling convention."""
    return tiled_matmul(x, w, block_m=block_m, block_k=block_k, block_n=block_n)


# Run auto-tuning (reduced iterations to keep the CPU demo fast; bump on TPU).
report = tune(
    kernel_fn=kernel_fn,
    input_fn=input_fn,
    config=config,
    strategy="grid",
    n_warmup=2,
    n_repeat=5,
)

Auto-tuning 18 configurations...
[1/18] Benchmarking: {'block_m': 128, 'block_k': 128, 'block_n': 128}
  -> median: 0.434 ms
[2/18] Benchmarking: {'block_m': 128, 'block_k': 128, 'block_n': 256}
  -> median: 0.315 ms
[3/18] Benchmarking: {'block_m': 128, 'block_k': 128, 'block_n': 512}
  -> median: 0.298 ms
[4/18] Benchmarking: {'block_m': 128, 'block_k': 256, 'block_n': 128}
  -> median: 0.340 ms
[5/18] Benchmarking: {'block_m': 128, 'block_k': 256, 'block_n': 256}
  -> median: 0.267 ms
[6/18] Benchmarking: {'block_m': 128, 'block_k': 256, 'block_n': 512}
  -> median: 0.254 ms
[7/18] Benchmarking: {'block_m': 256, 'block_k': 128, 'block_n': 128}
  -> median: 0.324 ms
[8/18] Benchmarking: {'block_m': 256, 'block_k': 128, 'block_n': 256}
  -> median: 0.268 ms
[9/18] Benchmarking: {'block_m': 256, 'block_k': 128, 'block_n': 512}
  -> median: 0.231 ms
[10/18] Benchmarking: {'block_m': 256, 'block_k': 256, 'block_n': 128}
  -> median: 0.292 ms
[11/18] Benchmarking: {'block_m': 256, 'block_

## 3. Analyze results

In [5]:
# Best and worst configs
best = report.best(3)
worst = report.worst(3)

print("Top 3 fastest configs:")
for i, r in enumerate(best):
    print(f"  {i+1}. {r.config} -> {r.median_ms:.3f} ms")

print("\nWorst 3 configs:")
for i, r in enumerate(worst):
    print(f"  {i+1}. {r.config} -> {r.median_ms:.3f} ms")

print(f"\nSpeedup range: {report.speedup_range:.2f}x")
print("(On TPU, this would typically be 3-5x)")

Top 3 fastest configs:
  1. {'block_m': 256, 'block_k': 256, 'block_n': 512} -> 0.204 ms
  2. {'block_m': 512, 'block_k': 128, 'block_n': 512} -> 0.207 ms
  3. {'block_m': 512, 'block_k': 128, 'block_n': 256} -> 0.216 ms

Worst 3 configs:
  1. {'block_m': 128, 'block_k': 128, 'block_n': 128} -> 0.434 ms
  2. {'block_m': 128, 'block_k': 256, 'block_n': 128} -> 0.340 ms
  3. {'block_m': 256, 'block_k': 128, 'block_n': 128} -> 0.324 ms

Speedup range: 2.13x
(On TPU, this would typically be 3-5x)


## 4. Generate performance heatmaps

Heatmaps are the visual centerpiece — they show how parameter choices affect performance.

In [6]:
# Heatmap: block_m vs block_n
fig = report.heatmap(
    "block_m",
    "block_n",
    metric="median_ms",
    title=f"MatMul Latency (ms) — {M}x{K}x{N}",
    cmap="YlOrRd_r",
)
plt.show()
print("Yellow = fast, Red = slow")

Yellow = fast, Red = slow


In [7]:
# Heatmap: block_m vs block_k
fig = report.heatmap(
    "block_m",
    "block_k",
    metric="median_ms",
    title="MatMul Latency by block_m vs block_k",
    cmap="YlOrRd_r",
)
plt.show()

## 5. Export results

In [8]:
import json
import tempfile
from pathlib import Path

tmpdir = tempfile.mkdtemp()

# JSON export
json_path = Path(tmpdir) / "matmul_results.json"
report.to_json(json_path)

# CSV export
csv_path = Path(tmpdir) / "matmul_results.csv"
report.to_csv(csv_path)

# Heatmap to file
heatmap_path = Path(tmpdir) / "heatmap.png"
report.heatmap("block_m", "block_n", save_path=heatmap_path)

# Show JSON structure
with open(json_path) as f:
    data = json.load(f)
print(f"Exported {len(data)} results to JSON")
print("\nSample entry:")
print(json.dumps(data[0], indent=2))

print(f"\nFiles saved to: {tmpdir}")

Exported 18 results to JSON

Sample entry:
{
  "config_block_m": 256,
  "config_block_k": 256,
  "config_block_n": 512,
  "median_ms": 0.20419999998466665,
  "mean_ms": 0.2092200000106459,
  "std_ms": 0.02339518837659038,
  "min_ms": 0.18451000005370588,
  "max_ms": 0.25118999997175706
}

Files saved to: /tmp/tmpj6iwnj7j


## 6. Random search for large spaces

When the search space is too large for exhaustive grid search, use random sampling.

In [9]:
from pallas_forge.tune.search import RandomSearch

# Larger TPU-valid search space — block_k/block_n must be >= 128.
large_config = TuneConfig.from_dict(
    {
        "block_m": [128, 256, 512, 1024],
        "block_k": [128, 256, 512, 1024],
        "block_n": [128, 256, 512, 1024],
    }
)

print(f"Total space: {large_config.total_combinations} configs")

# Sample 10 random configs
random_strategy = RandomSearch(n_trials=10, seed=42)
sampled = random_strategy.generate(large_config)
print(f"Sampled: {len(sampled)} configs")

for c in sampled:
    print(f"  {c}")

Total space: 64 configs
Sampled: 10 configs
  {'block_m': 128, 'block_k': 128, 'block_n': 512}
  {'block_m': 256, 'block_k': 256, 'block_n': 256}
  {'block_m': 128, 'block_k': 128, 'block_n': 1024}
  {'block_m': 128, 'block_k': 128, 'block_n': 128}
  {'block_m': 256, 'block_k': 256, 'block_n': 128}
  {'block_m': 256, 'block_k': 1024, 'block_n': 256}
  {'block_m': 1024, 'block_k': 512, 'block_n': 128}
  {'block_m': 256, 'block_k': 1024, 'block_n': 512}
  {'block_m': 512, 'block_k': 256, 'block_n': 256}
  {'block_m': 512, 'block_k': 128, 'block_n': 128}


## 7. Custom kernel tuning — SwiGLU example

In [10]:
# Tune SwiGLU — block_n must be >= 128; block_m can be any multiple of 8.
swiglu_config = TuneConfig.from_dict(
    {
        "block_m": [32, 64, 128],
        "block_n": [128, 256],
    }
)

BS, D, FFN = 128, 128, 256


def swiglu_input_fn(cfg):
    key = jax.random.PRNGKey(0)
    k1, k2, k3 = jax.random.split(key, 3)
    return (
        jax.random.normal(k1, (BS, D), dtype=jnp.float32),
        jax.random.normal(k2, (D, FFN), dtype=jnp.float32),
        jax.random.normal(k3, (D, FFN), dtype=jnp.float32),
    )


def swiglu_kernel_fn(x, w_gate, w_up, *, block_m, block_n):
    return fused_swiglu(x, w_gate, w_up, block_m=block_m, block_n=block_n)


swiglu_report = tune(
    kernel_fn=swiglu_kernel_fn,
    input_fn=swiglu_input_fn,
    config=swiglu_config,
    n_warmup=2,
    n_repeat=5,
)

fig = swiglu_report.heatmap(
    "block_m",
    "block_n",
    metric="median_ms",
    title=f"SwiGLU Latency (ms) — {BS}x{D}x{FFN}",
)
plt.show()

Auto-tuning 6 configurations...
[1/6] Benchmarking: {'block_m': 32, 'block_n': 128}
  -> median: 0.174 ms
[2/6] Benchmarking: {'block_m': 32, 'block_n': 256}
  -> median: 0.190 ms
[3/6] Benchmarking: {'block_m': 64, 'block_n': 128}
  -> median: 0.185 ms
[4/6] Benchmarking: {'block_m': 64, 'block_n': 256}
  -> median: 0.170 ms
[5/6] Benchmarking: {'block_m': 128, 'block_n': 128}
  -> median: 0.177 ms
[6/6] Benchmarking: {'block_m': 128, 'block_n': 256}
  -> median: 0.183 ms

Best config: {'block_m': 64, 'block_n': 256} -> 0.170 ms
Speedup range: 1.1x


## 8. YAML config files

For reproducible experiments, define configs in YAML and version-control them.

In [11]:
import tempfile

yaml_content = """
block_m: [128, 256, 512]
block_n: [128, 256, 512]
block_k: [128, 256]
"""

# Write to temp file
with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(yaml_content)
    yaml_path = f.name

# Load from YAML
yaml_config = TuneConfig.from_yaml(yaml_path)
print(f"Loaded from YAML: {yaml_config.param_names}")
print(f"Total combinations: {yaml_config.total_combinations}")

Loaded from YAML: ['block_m', 'block_n', 'block_k']
Total combinations: 18


## Key takeaways

1. **`tune()`** is the single entry point — pass any kernel + input_fn + config space
2. **Heatmaps** reveal which parameters matter most (on TPU, block_m and block_n typically dominate)
3. **Grid search** for small spaces, **random search** for large ones
4. **YAML configs** enable reproducible experiments
5. On CPU, timing differences are small; on TPU, expect 3-5x swings between best and worst configs

### Next steps (on TPU)

- Run with larger matrices (2048x2048+) on TPU to see real performance differences
- Add `flops_fn` and `bytes_fn` to compute TFLOPS and bandwidth
- Generate roofline charts to classify each config as compute-bound or memory-bound
- Capture XProf traces for top-N configs to understand hardware utilization